[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# SQLite and PostgreSQL &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup: one model, two `Database` objects with nothing named, and
the helpers that compile SQL without connecting. Run it first. The tasks can be run in any order.


In [1]:
import re
import tempfile
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (CharField, IntegerField, Model, OperationalError, PostgresqlDatabase,
                    SqliteDatabase, TextField, fn)
from playhouse.db_url import connect as connect_url
from playhouse.pool import PooledSqliteDatabase
from playhouse.postgres_ext import TSVectorField

WORK = Path(tempfile.mkdtemp(prefix="backends-"))                   # for the one section that needs files

sqlite = SqliteDatabase(None)                                       # named later, or never
postgres = PostgresqlDatabase(None)


class Note(Model):
    """One models module, for both backends."""

    title = CharField(max_length=80)
    body = TextField()
    reads = IntegerField(default=0)

    class Meta:
        database = sqlite


def compiled(build, database):
    """The SQL a query would send under a database, with nothing connected.

    `build` is a function rather than a query, because a query remembers the database it was
    built against: building it outside this block would compile it for the wrong backend.
    """
    with database.bind_ctx([Note]):
        return " ".join(build().sql()[0].split())


def created(database):
    """The CREATE TABLE peewee would emit for Note under a database."""
    with database.bind_ctx([Note]):
        return Note._schema._create_table().query()[0]


def refused(message):
    """A connection failure without the part of it that differs between operating systems."""
    return re.sub(r"(port \d+ failed):.*", r"\1: ...", message.strip().splitlines()[0])


print("peewee", peewee.__version__)
print("nothing is connected:", sqlite.is_closed(), postgres.is_closed())


peewee 4.5.1
nothing is connected: True True


**1.** One table, two dialects.


In [2]:
for name, database in (("sqlite", sqlite), ("postgres", postgres)):
    print(f"{name}:")
    for column in created(database).split("(", 1)[1].rstrip(")").split(", "):
        print("   ", column)
    print()


sqlite:
    "id" INTEGER NOT NULL PRIMARY KEY
    "title" VARCHAR(80) NOT NULL
    "body" TEXT NOT NULL
    "reads" INTEGER NOT NULL

postgres:
    "id" SERIAL NOT NULL PRIMARY KEY
    "title" VARCHAR(80) NOT NULL
    "body" TEXT NOT NULL
    "reads" INTEGER NOT NULL



One difference, on one column: `INTEGER NOT NULL PRIMARY KEY` against `SERIAL NOT NULL PRIMARY KEY`.
`VARCHAR(80)`, `TEXT` and `INTEGER` are written the same way by both, so `CharField(max_length=80)`
needed no translation at all.


**2.** Two more statements.


In [3]:
for label, build in (("delete", lambda: Note.delete().where(Note.reads == 0)),
                     ("select", lambda: Note.select(Note.title).limit(3))):
    print(f"{label}:")
    for name, database in (("sqlite  ", sqlite), ("postgres", postgres)):
        print(f"  {name} {compiled(build, database)}")


delete:
  sqlite   DELETE FROM "note" WHERE ("note"."reads" = ?)
  postgres DELETE FROM "note" WHERE ("note"."reads" = %s)
select:
  sqlite   SELECT "t1"."title" FROM "note" AS "t1" LIMIT ?
  postgres SELECT "t1"."title" FROM "note" AS "t1" LIMIT %s


Only the placeholder changed, in both. The `delete` has one, for the value on the right of the
comparison, and the `select` has one for the `LIMIT`, which is a bound value on both backends rather
than a number written into the statement.


**3.** Two databases from two URLs.


In [4]:
for url in (f"sqlite:///{WORK / 'tasks.db'}", "postgresql://someone@127.0.0.1:59999/tasks"):
    database = connect_url(url)
    print(f"  {url.split('://')[0]:<11} {type(database).__name__:<20} closed: {database.is_closed()}")


  sqlite      SqliteDatabase       closed: True
  postgresql  PostgresqlDatabase   closed: True


Both closed, including the one pointing at a port with nothing behind it. `connect_url` reads the
scheme and builds the matching class, and that is all it does.


**4.** Locked, and then not.


In [5]:
path = str(WORK / "tasks-lock.db")
maker = SqliteDatabase(path)
with maker.bind_ctx([Note]):
    maker.create_tables([Note])

one = SqliteDatabase(path, timeout=1)
two = SqliteDatabase(path, timeout=1)
one.connect()
two.connect()

one.begin()
one.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("a", "b", 0))
try:
    two.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("b", "b", 0))
except OperationalError as error:
    print("while the first transaction is open:", error)

one.commit()                                                        # the first writer finishes
two.execute_sql("INSERT INTO note (title, body, reads) VALUES (?, ?, ?)", ("b", "b", 0))
print("after it commits: the second writer got in")
print("rows:", one.execute_sql("SELECT COUNT(*) FROM note").fetchone()[0])
one.close()
two.close()


while the first transaction is open: database is locked
after it commits: the second writer got in
rows: 2


True

The same two connections, the same two statements, and the only thing that changed is whether the
first transaction was still open. That is what makes this hard to reproduce on demand and easy to
meet under load.


**5.** The statement that will not run here.


In [6]:
class Searchable(Model):
    body = TextField()
    search = TSVectorField()

    class Meta:
        database = sqlite


with sqlite.bind_ctx([Searchable]):
    print("table: ", Searchable._schema._create_table().query()[0])
    for statement in Searchable._schema._create_indexes():
        print("index: ", statement.query()[0])


table:  CREATE TABLE IF NOT EXISTS "searchable" ("id" INTEGER NOT NULL PRIMARY KEY, "body" TEXT NOT NULL, "search" TSVECTOR NOT NULL)
index:  CREATE INDEX IF NOT EXISTS "searchable_search" ON "searchable" USING GIN ("search")


The table would be accepted by SQLite, which gives a column type it does not know an affinity and
stores the value anyway. The index would not: `USING GIN` is PostgreSQL's, and nothing was run here
to find that out.


**6.** A connection, borrowed and returned.


In [7]:
borrowed = PooledSqliteDatabase(str(WORK / "tasks-pool.db"), max_connections=2, stale_timeout=30)

with borrowed.bind_ctx([Note]):
    print("at the start  -> in use:", len(borrowed._in_use), "| idle:", len(borrowed._connections))
    borrowed.connect()
    borrowed.create_tables([Note])
    print("while open    -> in use:", len(borrowed._in_use), "| idle:", len(borrowed._connections))
    borrowed.close()
    print("after close   -> in use:", len(borrowed._in_use), "| idle:", len(borrowed._connections))
    borrowed.dispose()
    print("after dispose -> in use:", len(borrowed._in_use), "| idle:", len(borrowed._connections))


at the start  -> in use: 0 | idle: 0
while open    -> in use: 1 | idle: 0
after close   -> in use: 0 | idle: 1
after dispose -> in use: 0 | idle: 0


`close` moved the connection from in use to idle rather than closing it, which is the whole point of
a pool. `dispose` is what actually closes them, and it is the call a forked child owes the pool it
inherited.


---

&#8592; **Back to:** [SQLite and PostgreSQL](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/11-sqlite-and-postgresql.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
